In [ ]:
!pip -q install fsspec aiohttp dask[dataframe] pyarrow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.6/221.6 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.3/377.3 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 118.0 MB/s eta 0:00:00


In [ ]:
import json, urllib.request, pandas as pd

record_id = "4650046"
with urllib.request.urlopen(f"https://zenodo.org/api/records/{record_id}") as r:
    rec = json.load(r)

def file_url(record_id, key):
    # construct a direct-download URL regardless of API shape
    return f"https://zenodo.org/records/{record_id}/files/{key}?download=1"

rows = []
for f in rec["files"]:
    rows.append({
        "name": f["key"],
        "size_bytes": f["size"],
        "size_gb": round(f["size"] / (1024**3), 2),
        "url": file_url(record_id, f["key"]),
        "checksum_md5": f.get("checksum", "").replace("md5:", "")
    })

files = pd.DataFrame(rows).sort_values("size_bytes", ascending=False)
files.head(10)

,name,size_bytes,size_gb,url,checksum_md5
9,youtube_comments.tsv.gz,77219676839,71.92,https://zenodo.org/records/4650046/files/youtu...,5baba0a24d738b49834790fe34ff9758
8,_raw_yt_metadata.jsonl.zst,14691107998,13.68,https://zenodo.org/records/4650046/files/_raw_...,d58f2e9964f2b73af4a2978beccde05f
6,yt_metadata_en.jsonl.gz,13636127630,12.70,https://zenodo.org/records/4650046/files/yt_me...,0514b2ee52ffaa2c9c27c539038feb60
5,yt_metadata_helper.feather,2831015738,2.64,https://zenodo.org/records/4650046/files/yt_me...,fce85455ea1b552fb834a338a6509b13
4,num_comments_authors.tsv.gz,1389789842,1.29,https://zenodo.org/records/4650046/files/num_c...,b5e126f814f0baf2942d5595c7f3054e
0,num_comments.tsv.gz,754585439,0.70,https://zenodo.org/records/4650046/files/num_c...,d250ca214b70a0f4f250b1ce6b1d73c1
7,_raw_df_timeseries.tsv.gz,653094906,0.61,https://zenodo.org/records/4650046/files/_raw_...,579e586802fd76a47a910437bdeb8bad
3,df_timeseries_en.tsv.gz,571058429,0.53,https://zenodo.org/records/4650046/files/df_ti...,689cf552e2a2c906ab7e41c01b2a8627
2,_raw_df_channels.tsv.gz,6404112,0.01,https://zenodo.org/records/4650046/files/_raw_...,83047b858a7c101b8c4e0539c8c143c2
1,df_channels_en.tsv.gz,5960728,0.01,https://zenodo.org/records/4650046/files/df_ch...,aa4d90892aeaae40089b5825c87607c8


In [ ]:
import pandas as pd, fsspec

def url_of(name):
    return files.loc[files["name"]==name, "url"].item()



LOAD CHANNEL METADATA

In [ ]:
name = "df_channels_en.tsv.gz"  # or any filename from your table
url = url_of(name)
wanted = ['author', 'video_id', 'likes', 'replies']  # tweak as needed
with fsspec.open(url, "rb") as fh:
    dfChannels = pd.read_csv(fh, sep="\t", compression="gzip",
                     nrows=1_000_000, usecols=None)
dfChannels.head()


,category_cc,join_date,channel,name_cc,subscribers_cc,videos_cc,subscriber_rank_sb,weights
0,Gaming,2010-04-29,UC-lHJZR3Gqxm24_Vd_AJ5Yw,PewDiePie,101000000,3956,3.0,2.087
1,Education,2006-09-01,UCbCmjCuTUZos6Inko4u57UQ,Cocomelon - Nursery ...,60100000,458,7.0,2.087
2,Entertainment,2006-09-20,UCpEhnqL0y41EpW2TvWAHD7Q,SET India,56018869,32661,8.0,2.087
3,Howto & Style,2016-11-15,UC295-Dw_tDNtZXFeAPAW6Aw,5-Minute Crafts,60600000,3591,9.0,2.087
4,Sports,2007-05-11,UCJ5v_MCY6GNUBTO8-D3XoAg,WWE,48400000,43421,11.0,2.087


LOAD VIDEO METADATA FEATHER


In [ ]:
# 1) Download once to local disk (seekable) with retries + progress
import os, hashlib, requests, shutil, sys

URL = "https://zenodo.org/records/4650046/files/yt_metadata_helper.feather?download=1"
LOCAL = "/content/yt_metadata_helper.feather"


if not os.path.exists(LOCAL):
    with requests.get(URL, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(LOCAL, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk); done += len(chunk)
                    if total: print(f"\r{done/total:.1%}", end=""); sys.stdout.flush()




100.0%

In [ ]:
cols = ["display_id", "channel_id"]
videoDf = pd.read_feather(LOCAL, columns=cols, dtype_backend="pyarrow")

In [ ]:
videoDf.head()

,display_id,channel_id
0,SBqSc91Hn9g,UCzWrhkg9eK5I8Bm3HfV-unA
1,UuugEl86ESY,UCzWrhkg9eK5I8Bm3HfV-unA
2,oB4c-yvnbjs,UCzWrhkg9eK5I8Bm3HfV-unA
3,ZaV-gTCMV8E,UCzWrhkg9eK5I8Bm3HfV-unA
4,cGvL7AvMfM0,UCzWrhkg9eK5I8Bm3HfV-unA


CREATE DICTIONARY FROM VIDEO ID TO CHANNEL ID

In [ ]:
v2c = dict(zip(videoDf.display_id.values, videoDf.channel_id.values))


CHOOSE SUBSET OF CHANNELS TO WORK ON

In [ ]:
channelSubset = dict(zip(dfChannels.channel.values, dfChannels.subscribers_cc.values > 200_000))

In [ ]:
channelSubsetDf = dfChannels[dfChannels['channel'].map(channelSubset)]
channelSubsetDf.to_csv('nodes.csv')

In [ ]:
channelSubsetDf

,category_cc,join_date,channel,name_cc,subscribers_cc,videos_cc,subscriber_rank_sb,weights
0,Gaming,2010-04-29,UC-lHJZR3Gqxm24_Vd_AJ5Yw,PewDiePie,101000000,3956,3.0,2.0870
1,Education,2006-09-01,UCbCmjCuTUZos6Inko4u57UQ,Cocomelon - Nursery ...,60100000,458,7.0,2.0870
2,Entertainment,2006-09-20,UCpEhnqL0y41EpW2TvWAHD7Q,SET India,56018869,32661,8.0,2.0870
3,Howto & Style,2016-11-15,UC295-Dw_tDNtZXFeAPAW6Aw,5-Minute Crafts,60600000,3591,9.0,2.0870
4,Sports,2007-05-11,UCJ5v_MCY6GNUBTO8-D3XoAg,WWE,48400000,43421,11.0,2.0870
...,...,...,...,...,...,...,...,...
28338,Entertainment,2015-05-18,UCmn56iouEYh1XDCgAR5VKGg,Skye Crew,201015,49,91704.0,3.8570
28362,Gaming,2009-11-18,UCPG-oduOIpiMscw0_Q6_QqQ,A10Games,200742,124,91798.0,3.8155
28367,Gaming,2015-01-07,UC3oHeWsu0w2I_4JnwHTz-Tw,Kolento,202000,2372,91813.0,3.8095
28527,Entertainment,2014-09-22,UCuw8yh-DJxouJl_ULQS_IBA,YoutubeBassBooster,216000,61,92255.0,3.8815


LOAD COMMENTS FROM SUBSET AND CREATE EDGES

In [ ]:
import pyarrow.feather as feather
from itertools import combinations
from collections import defaultdict, Counter
from tqdm import tqdm
import pickle, os, json

COMMENTS_URL = "youtube_comments.tsv.gz"

EDGES_OUT = "/content/channel_edges.csv"
CHUNKSIZE = 1_000_000
MAX_ROWS  = 150_000_000
PER_USER_CAP = 20
MIN_CHANS_FOR_PAIRS = 2

##CHECKPOINT LOGIC
CHECKPOINT_PATH = "/content/edges_checkpoint.pkl"
STATE_PATH = "/content/state.json"
CHECKPOINT_EVERY = 5

if os.path.exists(CHECKPOINT_PATH):
    print("Resuming from checkpoint...")
    with open(CHECKPOINT_PATH, "rb") as f:
        edges_counter = pickle.load(f)
else:
    edges_counter = Counter()

if os.path.exists(STATE_PATH):
    with open(STATE_PATH) as f:
        rows_seen = json.load(f)["rows_seen"]
else:
    rows_seen = 0

chunk_idx = 0
user_channels = defaultdict(set)

with fsspec.open(url_of(COMMENTS_URL), "rb") as fh:
    reader = pd.read_csv(
        fh,
        sep="\t",
        compression="gzip",
        usecols=["author", "video_id"],
        dtype=str,
        chunksize=CHUNKSIZE
    )

    for chunk in tqdm(reader, desc="Streaming comments in chunks"):
        chunk_idx += 1

        if MAX_ROWS and rows_seen >= MAX_ROWS:
            break

        if MAX_ROWS:
            remaining = MAX_ROWS - rows_seen
            if len(chunk) > remaining:
                chunk = chunk.iloc[:remaining]

        # map video -> channel; drop rows missing map or author
        chunk["channel_id"] = chunk["video_id"].map(v2c)
        chunk = chunk.dropna(subset=["author", "channel_id"])
        chunk = chunk[chunk['channel_id'].map(channelSubset)]

        # update capped per-user channel sets
        for author, col in chunk.groupby("author")["channel_id"]:
            s = user_channels[author]
            if len(s) < PER_USER_CAP:
                for ch in col.unique():
                    if len(s) < PER_USER_CAP:
                        s.add(ch)

        rows_seen += len(chunk)

        # periodic flush to keep memory stable
        if len(user_channels) > 500_000:
            for chans in user_channels.values():
                if len(chans) >= MIN_CHANS_FOR_PAIRS:
                    for a, b in combinations(sorted(chans), 2):
                        edges_counter[(a, b)] += 1
            user_channels.clear()

        if chunk_idx % CHECKPOINT_EVERY == 0:
          with open(CHECKPOINT_PATH, "wb") as f:
              pickle.dump(edges_counter, f)

          temp_df = pd.DataFrame(((a, b, w) for (a, b), w in edges_counter.items()),
                           columns=["src", "dst", "weight"])
          temp_df.to_csv('edges_checkpoint.csv', index=False)
          with open(STATE_PATH, "w") as f:
              json.dump({"rows_seen": rows_seen}, f)
          print(f"[checkpoint] {rows_seen:,} rows processed, checkpoint saved.")


# final flush
for chans in user_channels.values():
    if len(chans) >= MIN_CHANS_FOR_PAIRS:
        for a, b in combinations(sorted(chans), 2):
            edges_counter[(a, b)] += 1
user_channels.clear()

print(f"Processed ~{rows_seen:,} comment rows; unique edges: {len(edges_counter):,}")

edges_df = pd.DataFrame(((a, b, w) for (a, b), w in edges_counter.items()),
                        columns=["src", "dst", "weight"])
edges_df.to_csv(EDGES_OUT, index=False)
print(f"Wrote edges → {EDGES_OUT}")


Streaming comments in chunks: 5it [04:43, 54.90s/it]

[checkpoint] 3,973,180 rows processed, checkpoint saved.


Streaming comments in chunks: 10it [08:57, 51.28s/it]

[checkpoint] 8,002,932 rows processed, checkpoint saved.


Streaming comments in chunks: 15it [13:26, 53.95s/it]

...

Streaming comments in chunks: 120it [1:58:12, 69.92s/it]

[checkpoint] 96,157,506 rows processed, checkpoint saved.


Streaming comments in chunks: 125it [2:03:40, 72.69s/it]

[checkpoint] 100,133,003 rows processed, checkpoint saved.


Streaming comments in chunks: 129it [2:07:49, 66.67s/it]